In [ ]:
# Import TensorFlow
import tensorflow as tf

# Helper libraries
import numpy as np
import os

print(tf.__version__)

In [2]:
import json
import yaml
import psutil
import datetime 
from astra.src.transformer import AstraNet, contrastive_train
from astra.src.preprocessing import contrastive_data_loader
from astra.src.loss import nt_xent_loss 
from astra.src.scheduler import CustomSchedule

In [ ]:
num_gpus = 0
# Detect available GPUs
gpus = tf.config.experimental.list_physical_devices('GPU')
gpus_to_use = gpus[:num_gpus]
# Make only the selected GPUs visible to TensorFlow
tf.config.experimental.set_visible_devices(gpus_to_use, 'GPU')
print(f"Using {len(gpus_to_use)} specified GPU(s).")

In [4]:
path_config = "/media3/majumder/astra/config/contrastive-loss_triplet.yaml"

In [5]:

with open(path_config, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
# If the list of devices is not specified in
# `tf.distribute.MirroredStrategy` constructor, they will be auto-detected.
# strategy = tf.distribute.MirroredStrategy()
strategy = tf.distribute.get_strategy()
print('Number of devices: {}'.format(strategy.num_replicas_in_sync))

In [7]:
hparams = {
            
            "model_params": {
                "n_views": config['n_views'], 
                "num_layers": config['num_layers'], "d_model": config['d_model'], "num_heads": config['num_heads'],
                "dff": config['dff'], "projection_dim": config['projection_dim'], "rate": config['rate'], "mjd": config['mjd'],
                "use_band_info": config['use_band_info'], "base": config['base'], "use_drop": config['use_drop']
            },
            "training_params": {
                "epochs": config['epochs'], "patience": config['patience'], "initial_lr": config['initial_lr'],
                "use_custom_schedule": config['use_custom_schedule'], "warmup_steps": config['warmup_steps'],
                "temperature": config['temperature'], "batch_size": config['batch_size']
            },
            "data_params": {
                "buffer_size": config['buffer_size'],
                "apply_white_noise": config['apply_white_noise'], 
                "noise_levels": config['noise_levels'],
                "apply_binning": config['apply_binning'], 
                "apply_outlier": config['apply_outlier'],
                "maxlens": config['maxlens'], "bin_widths": config['bin_widths'], 
                "drop_rates": config['drop_rates'],
                "path_to_read": config['path_to_read'], "path_to_val": config['path_to_val']
                
            }
        }


In [ ]:
# BUFFER_SIZE = len(train_images)

# BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE = config['batch_size'] * strategy.num_replicas_in_sync
print('Global Batch Size: {}'.format(GLOBAL_BATCH_SIZE))
# EPOCHS = 10
build_seq_len = tf.cast(sum(config['maxlens'][0].values()), tf.int32)  # Final fixed length for sequences
    

In [ ]:
train_dataset = contrastive_data_loader(
                                        source=config['path_to_read'],
                                        n_views=config['n_views'], 
                                        seed=np.random.randint(1024), 
                                        batch_size=GLOBAL_BATCH_SIZE,
                                        build_seq_len=build_seq_len,
                                        apply_white_noise=config['apply_white_noise'],
                                        noise_levels=config['noise_levels'],
                                        apply_binning=config['apply_binning'],
                                        apply_outlier=config['apply_outlier'],
                                        maxlens=config['maxlens'],
                                        bin_widths=config['bin_widths'],
                                        drop_rates=config['drop_rates'],
                                        buffer_size=config['buffer_size']
                                    )
# ===============================================================================
# --- Distribute the datasets using the strategy ---
#
distributed_train_dataset = strategy.experimental_distribute_dataset(train_dataset)
# -------------------------------------------------------------------------------
print("\n\nData loader ready.")

In [ ]:
valid_dataset = contrastive_data_loader(
                                        source=config['path_to_read'],
                                        n_views=config['n_views'], 
                                        seed=np.random.randint(1024), 
                                        batch_size=GLOBAL_BATCH_SIZE,
                                        build_seq_len=build_seq_len,
                                        apply_white_noise=config['apply_white_noise'],
                                        noise_levels=config['noise_levels'],
                                        apply_binning=config['apply_binning'],
                                        apply_outlier=config['apply_outlier'],
                                        maxlens=config['maxlens'],
                                        bin_widths=config['bin_widths'],
                                        drop_rates=config['drop_rates'],
                                        buffer_size=max(config['buffer_size'] // 4, 100) # Smaller buffer for validation
                                        )
# -------------------------------------------------------------------------------
# --- Distribute the datasets using the strategy ---
#
distributed_val_dataset = strategy.experimental_distribute_dataset(valid_dataset)

In [ ]:
with strategy.scope():
    # ===============================================
    # Instantiate Model (SAME AS BEFORE, just indented)
    model = AstraNet(
        num_layers=hparams["model_params"]["num_layers"],
        d_model=hparams["model_params"]["d_model"],
        base=hparams["model_params"]["base"],
        num_heads=hparams["model_params"]["num_heads"],
        dff=hparams["model_params"]["dff"],
        rate=hparams["model_params"]["rate"],
        mjd=hparams["model_params"]["mjd"],
        use_drop=hparams["model_params"]["use_drop"],
        use_band_info=hparams["model_params"]["use_band_info"],
        projection_dim=hparams["model_params"]["projection_dim"] # Pass projection dim
    )

    # Instantiate Optimizer inside the scope
    if hparams["training_params"]['use_custom_schedule']:
        # Note: You need d_model from hparams here
        d_model = hparams["model_params"]["d_model"] 
        warmup_steps = hparams["training_params"]["warmup_steps"]
        custom_lr = CustomSchedule(d_model, warmup_steps=warmup_steps)
        optimizer = tf.keras.optimizers.Adam(learning_rate=custom_lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=hparams['initial_lr'])
    # --- END OF CHANGE ---

    # build_seq_len = tf.cast(sum(hparams['maxlens'][0].values()), tf.int32)  # Final fixed length for sequences
    dummy_input = {
        'input': tf.zeros((1, build_seq_len, 1), dtype=tf.float32),
        'times': tf.zeros((1, build_seq_len, 1), dtype=tf.float32),
        'band_info': tf.zeros((1, build_seq_len, 1), dtype=tf.float32),
        'mask': tf.zeros((1, build_seq_len), dtype=tf.float32) # Mask shape (batch, seq_len)
    }
    # Perform the dummy call
    _ = model(dummy_input,  training=False)
    # Print the model summary
    model.summary()

In [12]:
# ==========================================================
# 1. DEFINE THE DISTRIBUTED TRAINING STEP
# ==========================================================
@tf.function
def distributed_train_step(model, optimizer, strategy, global_batch_size, dist_inputs, temperature):
    """
    Performs one distributed training step.
    """
    def train_step(inputs):
        # Unpack the views for this replica's mini-batch
        *views_batch, = inputs

        with tf.GradientTape() as tape:
            # Get projections from the model
            z_views = [model(view, training=True) for view in views_batch]
            # Calculate the loss for this replica's mini-batch
            loss = nt_xent_loss(*z_views, temperature=temperature)
            # IMPORTANT: Scale the loss by the GLOBAL batch size.
            # This ensures the gradients are correctly averaged, not summed.
            scaled_loss = tf.nn.compute_average_loss(loss, global_batch_size=global_batch_size)

            

        grads = tape.gradient(scaled_loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return tf.reduce_mean(loss)
    
    per_replica_losses = strategy.run(train_step, args=(dist_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)


@tf.function
def distributed_validation_step(model, strategy, dist_inputs, temperature):
    """
    Performs one distributed validation step.
    """
    def valid_step(inputs):
        *views_batch, = inputs
        z_views = [model(view, training=False) for view in views_batch] 
        loss = nt_xent_loss(*z_views, temperature=temperature)
        return tf.reduce_mean(loss)
    
    per_replica_losses = strategy.run(valid_step, args=(dist_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_losses, axis=None)

In [ ]:
for epoch in range(config['epochs']):
    # TRAIN LOOP
    total_train_loss = 0.0
    num_train_batches = 0
    model.trainable = True # Ensure model is trainable
    for x in distributed_train_dataset:
        # Call the distributed train step function to get the loss for this global batch
        current_train_loss = distributed_train_step(model, optimizer, strategy, GLOBAL_BATCH_SIZE, x, config['temperature'])
        total_train_loss += current_train_loss
        num_train_batches += 1
    
    mean_epoch_train_loss = total_train_loss / num_train_batches

    
    total_val_loss = 0.0
    num_val_batches = 0
    model.trainable = False # Set model to non-trainable for validation

    # TEST LOOP
    for x in distributed_val_dataset:
        #
        current_val_loss = distributed_validation_step(model, strategy, x, config['temperature'])
        total_val_loss += current_val_loss
        num_val_batches += 1

    mean_epoch_val_loss = total_val_loss / num_val_batches
        


    template = ("Epoch {}, Train-Loss: {}, Validation-Loss: {}")
    print(template.format(epoch + 1, mean_epoch_train_loss.numpy(), mean_epoch_val_loss.numpy()))
                        